# Donchian + VWAP — optional ATR trail test

Same entries (Donchian 55 + VWAP 20), exit via **chandelier ATR trail**.

This notebook loads the precomputed grid from `pipeline/research/artifacts/donchian_atr_trail_grid.csv` (run `python -m pipeline.research.donchian_atr_grid` to refresh), then builds equity curves for the best OOS Sharpe params.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path(os.environ.get("ORBIS_REPO", "/workspace")).resolve()
if not (ROOT / "pipeline").exists():
    ROOT = Path.cwd().resolve()
    while ROOT.name and not (ROOT / "pipeline").exists():
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline.research.donchian_vwap_backtest import (
    _metrics,
    benchmark_returns,
    build_research_panels,
    is_oos_split,
    run_donchian_vwap_portfolio,
)

(is_start, is_end), (oos_start, oos_end) = is_oos_split()
full_start, full_end = is_start, oos_end
grid = pd.read_csv(ROOT / "pipeline/research/artifacts/donchian_atr_trail_grid.csv")
grid.head(10)

In [ ]:
best = grid.iloc[0]
p, m = int(best["atr_period"]), float(best["atr_mult"])
print(f"Best (OOS Sharpe): ATR period={p}, mult={m}")

panels, signals = build_research_panels()
daily_ret, _ = run_donchian_vwap_portfolio(
    full_start, full_end, exit_mode="donchian20", panels=panels, signals=signals
)
atr_ret, eq_atr = run_donchian_vwap_portfolio(
    full_start,
    full_end,
    exit_mode="atr_trail",
    atr_period=p,
    atr_mult=m,
    panels=panels,
    signals=signals,
)
spy_r = benchmark_returns(full_start, full_end, "SPY")

dc_m = _metrics("DC20 OOS", oos_start, oos_end, daily_ret)
atr_m = _metrics("ATR OOS", oos_start, oos_end, atr_ret)
print(f"OOS DC20 exit: {dc_m.total_return_pct:+.1f}% Sharpe {dc_m.sharpe:.2f} maxDD {dc_m.max_drawdown_pct:.1f}%")
print(f"OOS ATR exit: {atr_m.total_return_pct:+.1f}% Sharpe {atr_m.sharpe:.2f} maxDD {atr_m.max_drawdown_pct:.1f}%")

In [ ]:
def cum(ret):
    return (1 + ret.fillna(0)).cumprod() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(cum(daily_ret), label="Donchian-20 exit")
axes[0].plot(cum(atr_ret), label=f"ATR trail ({p}×{m})")
axes[0].plot(cum(spy_r.reindex(daily_ret.index).fillna(0)), label="SPY", alpha=0.7)
axes[0].axvline(pd.Timestamp(is_end), color="gray", ls="--")
axes[0].set_title("Full sample")
axes[0].legend(fontsize=8)

for r, lab in [(daily_ret, "DC20"), (atr_ret, f"ATR {p}×{m}"), (spy_r, "SPY")]:
    sub = r[(r.index >= pd.Timestamp(oos_start)) & (r.index <= pd.Timestamp(oos_end))]
    axes[1].plot(cum(sub), label=lab)
axes[1].set_title("OOS window")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()